# 4. Datareduktion
PCA och UMAP på de sex lyckofaktorerna från WHR 2019-2025.

In [139]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import umap.umap_ as umap

df = pd.read_csv("../data/cleaned_data.csv")

faktorer = [
    "Explained by: Log GDP per capita",
    "Explained by: Social support",
    "Explained by: Healthy life expectancy",
    "Explained by: Freedom to make life choices",
    "Explained by: Generosity",
    "Explained by: Perceptions of corruption"
]

faktor_etiketter = ["BNP", "Socialt stöd", "Hälsa", "Frihet", "Generositet", "Korruption"]

df_2019 = df[df["Year"] == 2019].set_index("Country name")[["Life evaluation (3-year average)"]]
df_2025 = df[df["Year"] == 2025].set_index("Country name")[["Life evaluation (3-year average)"]]
delta = (df_2025 - df_2019).rename(columns={"Life evaluation (3-year average)": "delta"})

df_X = df[df["Year"] == 2025].set_index("Country name")[faktorer]
df_X = df_X.join(delta).dropna()

X = df_X[faktorer].values
länder = df_X.index.tolist()
färger = df_X["delta"].values

print(f"X: {X.shape}")

X: (141, 6)


In [140]:
# Skala om alla 6 faktorer så att varje kolumn får medelvärde 0 och standardavvikelse 1.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Bekräfta: varje kolumn ska ha medelvärde ≈0 och std ≈1
print("Medelvärden:", X_scaled.mean(axis=0).round(3))
print("Std:        ", X_scaled.std(axis=0).round(3))


# Till presentation:  BNP har medelvärde ~1.27, Korruption ~0.15. 
# Utan skalning tror PCA att BNP är "viktigare" bara för att den har större tal. det är inte sant.

Medelvärden: [-0.  0. -0.  0. -0.  0.]
Std:         [1. 1. 1. 1. 1. 1.]


In [141]:
# Kör PCA på alla 6 dimensioner och titta på hur mycket varians varje principalkomponent "(PC)" förklarar.
pca_full = PCA(n_components=6)
pca_full.fit(X_scaled)

varians = pca_full.explained_variance_ratio_ * 100
kumulativ = np.cumsum(varians)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=[f"PC{i}" for i in range(1, 7)],
    y=varians,
    name="Per komponent",
    marker_color="#4fc3f7",
    text=[f"{v:.1f}%" for v in varians],
    textposition="outside",
    textfont=dict(color="white")
))

fig.add_trace(go.Scatter(
    x=[f"PC{i}" for i in range(1, 7)],
    y=kumulativ,
    name="Kumulativt",
    mode="lines+markers",
    line=dict(color="#ff7043", width=2.5),
    marker=dict(size=8)
))

fig.add_hline(y=80, line_dash="dash", line_color="#888888",
              annotation_text="80%", annotation_font_color="white")

fig.update_layout(
    title=dict(text="Scree plot – förklarad varians per principalkomponent",
               font=dict(color="white", size=16), x=0.5),
    xaxis=dict(title="Principalkomponent", title_font=dict(color="white"),
               tickfont=dict(color="white"), gridcolor="#444444"),
    yaxis=dict(title="Förklarad varians (%)", title_font=dict(color="white"),
               tickfont=dict(color="white"), gridcolor="#444444"),
    plot_bgcolor="#2d2d2d",
    paper_bgcolor="#2d2d2d",
    legend=dict(font=dict(color="white")),
    height=450
)

fig.show()
    
# Till presentation: Scree-plotten visar om 2 komponenter räcker för att fånga mönstret. 
# Om PC1+PC2 förklarar t.ex. 70% av variansen är en 2D-plot meningsfull.

In [142]:
# Reducera till 2 dimensioner och plotta varje land som en punkt. Färgkoda efter delta (grön = vinnare, röd = förlorare).
pca2 = PCA(n_components=2)
X_pca = pca2.fit_transform(X_scaled)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    mode="markers",
    marker=dict(
        size=8,
        color=färger,
        colorscale="RdYlGn",
        colorbar=dict(title="Delta", tickfont=dict(color="white"),
                      title_font=dict(color="white")),
        showscale=True
    ),
    text=[f"{land}<br>Delta: {d:.3f}" for land, d in zip(länder, färger)],
    hovertemplate="%{text}<extra></extra>"
))

fig.update_layout(
    title=dict(text="PCA – länder i 2D-rum (färg = förändring 2019→2025)",
               font=dict(color="white", size=16), x=0.5),
    xaxis=dict(title=f"PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}% varians)",
               title_font=dict(color="white"), tickfont=dict(color="white"),
               gridcolor="#444444"),
    yaxis=dict(title=f"PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}% varians)",
               title_font=dict(color="white"), tickfont=dict(color="white"),
               gridcolor="#444444"),
    plot_bgcolor="#2d2d2d",
    paper_bgcolor="#2d2d2d",
    height=600
)

fig.show()

# Till presentation: Det är den centrala visualiseringen. visar om vinnare och förlorare grupperar sig i PCA-rummet.

In [143]:
# Titta på PCA loadings. hur mycket varje ursprunglig faktor bidrar till PC1 och PC2?
# Visa hur mycket varje faktor bidrar till PC1 och PC2.
loadings = pd.DataFrame(
    pca2.components_.T,
    index=faktor_etiketter,
    columns=["PC1", "PC2"]
)

fig = go.Figure()

fig.add_trace(go.Bar(
    y=loadings.index,
    x=loadings["PC1"],
    orientation="h",
    name="PC1",
    marker_color="#4fc3f7",
    text=loadings["PC1"].round(2),
    textposition="outside",
    textfont=dict(color="white")
))

fig.add_trace(go.Bar(
    y=loadings.index,
    x=loadings["PC2"],
    orientation="h",
    name="PC2",
    marker_color="#ff7043",
    text=loadings["PC2"].round(2),
    textposition="outside",
    textfont=dict(color="white")
))

fig.add_vline(x=0, line_color="white", line_width=0.8)

fig.update_layout(
    title=dict(text="PCA loadings – faktorernas bidrag till PC1 och PC2",
               font=dict(color="white", size=16), x=0.5),
    xaxis=dict(title="Loading", title_font=dict(color="white"),
               tickfont=dict(color="white"), gridcolor="#444444"),
    yaxis=dict(tickfont=dict(color="white")),
    plot_bgcolor="#2d2d2d",
    paper_bgcolor="#2d2d2d",
    legend=dict(font=dict(color="white")),
    barmode="group",
    height=400
)

fig.show()

# Till presentation: Visar vilka faktorer som kännetecknar strukturellt starka/svaga länder.
# PC1 representerar den dominerande variationsriktningen; den faktor med högst loading där är den viktigaste.

In [144]:
# Kör UMAP på samma X_scaled och plotta på samma sätt som PCA.
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
X_umap = reducer.fit_transform(X_scaled)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=X_umap[:, 0],
    y=X_umap[:, 1],
    mode="markers",
    marker=dict(
        size=8,
        color=färger,
        colorscale="RdYlGn",
        colorbar=dict(title="Delta", tickfont=dict(color="white"),
                      title_font=dict(color="white")),
        showscale=True
    ),
    text=[f"{land}<br>Delta: {d:.3f}" for land, d in zip(länder, färger)],
    hovertemplate="%{text}<extra></extra>"
))

fig.update_layout(
    title=dict(text="UMAP – länder i 2D-rum (färg = förändring 2019→2025)",
               font=dict(color="white", size=16), x=0.5),
    xaxis=dict(title="UMAP-1", title_font=dict(color="white"),
               tickfont=dict(color="white"), gridcolor="#444444"),
    yaxis=dict(title="UMAP-2", title_font=dict(color="white"),
               tickfont=dict(color="white"), gridcolor="#444444"),
    plot_bgcolor="#2d2d2d",
    paper_bgcolor="#2d2d2d",
    height=600
)

fig.show()

# Till presentation: UMAP är bättre på att bevara lokala klusterstrukturer. 
# Jämförelsen mot PCA visar om det finns icke-linjära mönster som PCA missar.

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [145]:
# Kör UMAP tre gånger med olika n_neighbors och visa plottarna sida vid sida.
# n_neighbors styr hur "lokal" UMAP är. Lågt värde = mer fokus på närmaste grannar, högt värde = mer global struktur.
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=["n_neighbors=5", "n_neighbors=15", "n_neighbors=30"])

for col, n in enumerate([5, 15, 30], 1):
    emb = umap.UMAP(n_neighbors=n, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    fig.add_trace(go.Scatter(
        x=emb[:, 0], y=emb[:, 1],
        mode="markers",
        marker=dict(size=6, color=färger, colorscale="RdYlGn", showscale=(col == 3),
                    colorbar=dict(title="Delta", tickfont=dict(color="white"),
                                  title_font=dict(color="white"))),
        text=[f"{land}<br>Delta: {d:.3f}" for land, d in zip(länder, färger)],
        hovertemplate="%{text}<extra></extra>",
        showlegend=False
    ), row=1, col=col)

fig.update_layout(
    title=dict(text="UMAP – hyperparametertest (n_neighbors)",
               font=dict(color="white", size=16), x=0.5),
    plot_bgcolor="#2d2d2d",
    paper_bgcolor="#2d2d2d",
    height=500
)

for ann in fig.layout.annotations:
    ann.font.color = "white"

fig.update_xaxes(gridcolor="#444444", tickfont=dict(color="white"))
fig.update_yaxes(gridcolor="#444444", tickfont=dict(color="white"))

fig.show()

# Till presentation: n_neighbors styr balansen mellan lokal och global struktur. 
# Arbetsplanen kräver att ni testar 5, 15 och 30 och jämför. det visar att ni förstår algoritmen.
# n_neighbors=5 → tight lokala kluster. n_neighbors=30 → mer global struktur, liknar PCA mer.

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [146]:
# Svar på Följdfråga 1: finns det regionala mönster bland vinnare och förlorare?
# Vi mappar varje land till en FN-världsregion och räknar hur det gick per region.
import country_converter as coco

cc = coco.CountryConverter()
regioner = cc.convert(länder, to='UNregion', not_found='Okänd')

df_region = pd.DataFrame({
    'land':   länder,
    'region': regioner,
    'delta':  färger
})

# Summera per region: antal länder, antal förlorare och genomsnittligt delta
region_sammanfattning = (
    df_region
    .groupby('region')
    .agg(
        antal_länder=('delta', 'count'),
        antal_förlorare=('delta', lambda x: (x < 0).sum()),
        snitt_delta=('delta', 'mean')
    )
    .query('antal_länder >= 3')
    .sort_values('snitt_delta')
    .reset_index()
)

etikett = [
    f"Δ {d:.2f}  ({f} av {a} länder försämrades)"
    for d, f, a in zip(
        region_sammanfattning['snitt_delta'],
        region_sammanfattning['antal_förlorare'],
        region_sammanfattning['antal_länder']
    )
]

mask_pos = region_sammanfattning['snitt_delta'] >= 0
mask_neg = region_sammanfattning['snitt_delta'] < 0

# Bevarar sorteringsordningen (sämst längst ned, bäst längst upp) för båda traces
sorteringsordning = region_sammanfattning['region'].tolist()

fig = go.Figure()

# Positiva regioner — text utanför stapeln (till höger)
fig.add_trace(go.Bar(
    x=region_sammanfattning.loc[mask_pos, 'snitt_delta'],
    y=region_sammanfattning.loc[mask_pos, 'region'],
    orientation='h',
    marker_color='#66bb6a',
    text=[etikett[i] for i in region_sammanfattning.index[mask_pos]],
    textposition='outside',
    textfont=dict(color='white', size=11),
    showlegend=False
))

# Negativa regioner — text utanför stapeln (till vänster om stappelspetsen)
fig.add_trace(go.Bar(
    x=region_sammanfattning.loc[mask_neg, 'snitt_delta'],
    y=region_sammanfattning.loc[mask_neg, 'region'],
    orientation='h',
    marker_color='#ef5350',
    text=[etikett[i] for i in region_sammanfattning.index[mask_neg]],
    textposition='outside',
    textfont=dict(color='white', size=11),
    showlegend=False
))

fig.add_vline(x=0, line_color='white', line_width=1)

fig.update_layout(
    title=dict(
        text='Genomsnittlig lyckoförändring 2019→2025 per världsregion',
        font=dict(color='white', size=16),
        x=0.5
    ),
    xaxis=dict(
        title='Genomsnittligt delta',
        title_font=dict(color='white'),
        tickfont=dict(color='white'),
        gridcolor='#444444',
        range=[-0.75, 0.72]   # extra utrymme för text på båda sidor
    ),
    yaxis=dict(
        tickfont=dict(color='white'),
        categoryorder='array',
        categoryarray=sorteringsordning   # tvingar fram korrekt sorteringsordning
    ),
    plot_bgcolor='#2d2d2d',
    paper_bgcolor='#2d2d2d',
    barmode='overlay',
    height=520,
    margin=dict(r=20)
)

fig.show()

# Till presentation: Följdfråga 1 besvarad. Västra Afrika och Västeuropa är de regioner
# som i snitt försämrats mest. Östasien, Centralasien och Sydeuropa är tydliga vinnare.

In [147]:
# Svar på Följdfråga 2: vilken faktors förändring hänger starkast ihop med förändringen i lycka?
# För varje land beräknas skillnaden per faktor (2025 − 2019) och korreleras mot lycko-delta.
# Resultatet visar vilka faktorer som bäst predicerar om ett land förbättrade eller försämrade sin lyckopoäng.

# Delta per faktor: värdet 2025 minus värdet 2019 för samma land
df_faktor_2019 = df[df['Year'] == 2019].set_index('Country name')[faktorer]
df_faktor_2025 = df[df['Year'] == 2025].set_index('Country name')[faktorer]
delta_faktorer = (df_faktor_2025 - df_faktor_2019).dropna()
delta_faktorer.columns = faktor_etiketter

# # Beräkna lycko-delta (2025 − 2019) och säkerställ att det matchar samma länder som faktoranalysen
delta_lycka = (
    df[df['Year'] == 2025].set_index('Country name')['Life evaluation (3-year average)']
    - df[df['Year'] == 2019].set_index('Country name')['Life evaluation (3-year average)']
).reindex(delta_faktorer.index)

# Korrelation: hur starkt hänger varje faktors förändring ihop med lyckoförändringen?
korrelationer = delta_faktorer.corrwith(delta_lycka).sort_values(ascending=True)

# Färgkoda: grön om positiv korrelation, röd om negativ
korr_färger = ['#ef5350' if k < 0 else '#66bb6a' for k in korrelationer]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=korrelationer.values,
    y=korrelationer.index,
    orientation='h',
    marker_color=korr_färger,
    text=[f"{k:.2f}" for k in korrelationer.values],
    textposition='outside',
    textfont=dict(color='white', size=12)
))

fig.add_vline(x=0, line_color='white', line_width=1)

fig.update_layout(
    title=dict(
        text='Korrelation: förändring per faktor vs förändring i lycka (2019→2025)',
        font=dict(color='white', size=16),
        x=0.5
    ),
    xaxis=dict(
        title='Korrelation (Pearson r)',
        title_font=dict(color='white'),
        tickfont=dict(color='white'),
        gridcolor='#444444',
        range=[-0.35, 0.7]
    ),
    yaxis=dict(tickfont=dict(color='white')),
    plot_bgcolor='#2d2d2d',
    paper_bgcolor='#2d2d2d',
    height=400
)

fig.show()

# Till presentation: Följdfråga 2 besvarad direkt.
# Socialt stöd (r=0.54) och Frihet (r=0.38) är de faktorer vars förändring
# bäst förklarar om ett land vann eller förlorade i lycka.
# BNP och Hälsa är svagt negativt korrelerade — att vara rik garanterar inte förbättring.